In [ ]:
# ==========================================================
# NOTEBOOK 4
# CHATBOT DEPLOYMENT
#
# INPUT:
# Fine tuned model
#
# OUTPUT:
# Public Chatbot URL
#
# ==========================================================

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip uninstall -y torchao
!pip install torchao==0.16.0

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 1.2 MB/s eta 0:00:00


In [ ]:
import torchao
print(torchao.__version__)

0.16.0


In [ ]:
!pip install -q transformers
!pip install -q peft
!pip install -q gradio
!pip install -q accelerate
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.9 MB/s eta 0:00:00


In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/RetailBot_AI_Powered_ECommerce_Customer_Service_LLM"

MODEL_PATH = (
    f"{PROJECT_PATH}/model/retailbot_tinyllama/processed2"
)

# **Local/Temp Deployment for verification**

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto"
)

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/RetailBot_AI_Powered_ECommerce_Customer_Service_LLM/model/retailbot_tinyllama/processed2'. Use `repo_type` argument if needed.

In [ ]:
def chatbot(message, history):

    prompt = f"""<|user|>
{message}

<|assistant|>
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=55,
        temperature=0.2,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    if "<|assistant|>" in response:
        response = response.split("<|assistant|>")[-1]

    return response.strip()

In [ ]:
import gradio as gr
demo = gr.ChatInterface(
    fn=chatbot,
    title="Retail Assist AI ChatBot",
    description="E-Commerce Support Bot"
)

demo.launch(share=True)

In [ ]:
# demo.close()

# **Huggingface Deployment**

In [ ]:
MERGED_PATH = (
    f"{PROJECT_PATH}/model/retailbot_tinyllama_merged"
)

In [ ]:
# Merge modal for huggingface (Don't run again if want to skip if merged data is alredy saved in previous run)

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

model = PeftModel.from_pretrained(
    base_model,
    MODEL_PATH
)

merged_model = model.merge_and_unload()

merged_model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)

print("Merged model saved to:", MERGED_PATH)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ValueError: Can't find 'adapter_config.json' at '/content/drive/MyDrive/RetailBot_AI_Powered_ECommerce_Customer_Service_LLM/model/retailbot_tinyllama/processed2'

In [ ]:
import os

print(os.listdir(MERGED_PATH))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/RetailBot_AI_Powered_ECommerce_Customer_Service_LLM/model/retailbot_tinyllama_merged'

In [ ]:
!du -sh "{MERGED_PATH}"

du: cannot access '/content/drive/MyDrive/RetailBot_AI_Powered_ECommerce_Customer_Service_LLM/model/retailbot_tinyllama_merged': No such file or directory


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "/content/drive/MyDrive/RetailBot_AI_Powered_ECommerce_Customer_Service_LLM/model/retailbot_tinyllama_merged"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto"
)

print("Loaded successfully")

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/RetailBot_AI_Powered_ECommerce_Customer_Service_LLM/model/retailbot_tinyllama_merged'. Use `repo_type` argument if needed.

# Hugging Face Hub Login

In [3]:
!pip install -q huggingface_hub

In [5]:
from huggingface_hub import login

login()

In [ ]:
# create folder where modal will be save

from huggingface_hub import create_repo

create_repo(
    "deekshapal1/RetailBot-TinyLlama",
    repo_type="model",
    exist_ok=True
)

RepoUrl('https://huggingface.co/deekshapal1/RetailBot-TinyLlama', endpoint='https://huggingface.co', repo_type='model', repo_id='deekshapal1/RetailBot-TinyLlama')

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="/content/drive/MyDrive/RetailBot_AI_Powered_ECommerce_Customer_Service_LLM/model/retailbot_tinyllama_merged",
    repo_id="deekshapal1/RetailBot-TinyLlama",
    repo_type="model"
)

ValueError: Provided path: '/content/drive/MyDrive/RetailBot_AI_Powered_ECommerce_Customer_Service_LLM/model/retailbot_tinyllama_merged' is not a directory